# README

- **Author**: Codex
- **Created At**: 2026-10-13
- **Last Modified At**: 2026-10-13

---

## What does this file do?

這份 notebook 整理前一階段的假訂單資料，讓後面的分析可以直接使用。

先刪除 2 列完全相同的重複紀錄，再排除 3 筆購買件數缺漏、2 筆單價為負的訂單。這些問題是產生假資料時刻意加入的，因此這次直接排除，不另外補值。122 列資料最後保留 115 筆。

接著去除商品類別前後的空白、統一日期格式，並將購買件數轉成整數。最後新增訂單金額：

`order_amount_twd = quantity × unit_price_twd`

---

## What does this file take?

- **Source Data Sets**:
  1. `data/raw/demo_orders.csv`
     - Input key: `orders`
     - Description: 122 列原始訂單紀錄，共 5 欄。
     - Documentation: [原始資料說明](../raw/demo_orders_documentation.ipynb)
     - SHA-256: `fdf4f852be719af4c6bf4d7a8bb2246f3965cf90a6b0dbfd6153e7ae582d1abb`

---

## What does this file output?

- `data/processed/demo_orders_clean.csv`
  - Output key: `analysis`
  - Description: 115 筆整理後的訂單，新增訂單金額，共 6 欄。
  - Role: `analysis`
  - Documentation: [整理後資料說明](../processed/demo_orders_documentation.ipynb)
  - Expected schema and row-count rules: 115 列、6 欄；訂單編號不可重複；商品類別只有 A、B、C；金額等於件數乘以單價。
  - Verification status / SHA-256: 已重新讀取存出的 CSV，確認筆數、欄位、型態和數值與處理結果一致。檔案指紋為 `566d03f1701b16128fd119cc2776d775787816fb13f3d7e6d799988f6ce43104`。
  - Downstream finding input: `data/processed/demo_orders_clean.csv`

原始檔不會被修改。如果輸出檔已存在，程式會先比對內容；內容不同時會停止，避免蓋掉其他版本。

In [1]:
from pathlib import Path
import hashlib

import pandas as pd
from IPython.display import display

# 從 notebook 所在位置往上找專案，從根目錄開啟也能執行。
folder = Path.cwd().resolve()
for project_root in [folder, *folder.parents]:
    if (project_root / "AGENTS.md").is_file() and (project_root / "templates").is_dir():
        break
else:
    raise FileNotFoundError("找不到專案根目錄，請從專案內開啟 notebook。")

In [2]:
data_path = (project_root / "data/raw/demo_orders.csv").resolve()
expected_hash = "fdf4f852be719af4c6bf4d7a8bb2246f3965cf90a6b0dbfd6153e7ae582d1abb"

if data_path.parent != project_root / "data/raw":
    raise ValueError("資料路徑超出預期目錄。")
if not data_path.is_file():
    raise FileNotFoundError(f"找不到資料：{data_path.name}，請先產生資料。")

# 確認讀到的是這份文件記錄的資料版本。
file_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
if file_hash != expected_hash:
    raise ValueError("資料版本已改變，請先核對來源與說明文件。")

raw = pd.read_csv(data_path)
expected_columns = [
    "order_id", "order_date", "category", "quantity", "unit_price_twd"
]
assert list(raw.columns) == expected_columns
assert len(raw) == 122

In [3]:
# 只刪除整列相同的重複紀錄。
clean = raw.drop_duplicates().copy()
duplicate_count = len(raw) - len(clean)

# 這份假資料的缺值和負單價是刻意加入的，這次直接排除。
missing_quantity = clean["quantity"].isna()
invalid_price = clean["unit_price_twd"] <= 0
assert not (missing_quantity & invalid_price).any()
clean = clean.loc[~missing_quantity & ~invalid_price].copy()

clean["category"] = clean["category"].str.strip()
dates = pd.to_datetime(clean["order_date"], format="%Y-%m-%d")
clean["order_date"] = dates.dt.strftime("%Y-%m-%d")

# 先確認沒有小數，再轉成整數，避免轉型時把數量截掉。
assert (clean["quantity"] % 1 == 0).all()
clean["quantity"] = clean["quantity"].astype(int)
clean["order_amount_twd"] = clean["quantity"] * clean["unit_price_twd"]
clean = clean.sort_values("order_id").reset_index(drop=True)

assert duplicate_count == 2
assert missing_quantity.sum() == 3
assert invalid_price.sum() == 2
assert len(clean) == 115
assert clean["order_id"].is_unique
assert not clean.isna().any().any()
assert set(clean["category"]) == {"A", "B", "C"}
assert clean["quantity"].between(1, 5).all()
assert clean["unit_price_twd"].between(80, 300).all()

print(f"原始資料：{len(raw)} 列")
print(f"刪除重複：{duplicate_count} 列")
print(f"排除數量缺值：{missing_quantity.sum()} 筆")
print(f"排除非正單價：{invalid_price.sum()} 筆")
print(f"整理後：{len(clean)} 筆")

原始資料：122 列
刪除重複：2 列
排除數量缺值：3 筆
排除非正單價：2 筆
整理後：115 筆


In [4]:
output_path = (project_root / "data/processed/demo_orders_clean.csv").resolve()
if output_path.parent != project_root / "data/processed":
    raise ValueError("輸出路徑超出預期目錄。")

# 重跑時可接受相同結果，但不蓋掉內容不同的既有檔案。
csv_bytes = clean.to_csv(index=False, lineterminator="\n").encode("utf-8")
if output_path.exists():
    if output_path.read_bytes() != csv_bytes:
        raise FileExistsError("輸出檔已存在且內容不同，請先確認要保留的版本。")
else:
    with output_path.open("xb") as file:
        file.write(csv_bytes)

# 重新讀取存出的檔案，確認沒有少列、變更型態或改到數值。
saved = pd.read_csv(output_path)
pd.testing.assert_frame_equal(clean, saved)
assert (saved["order_amount_twd"] == saved["quantity"] * saved["unit_price_twd"]).all()
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == expected_hash

output_hash = hashlib.sha256(output_path.read_bytes()).hexdigest()
print(f"已核對 {output_path.name}，共 {len(saved)} 筆；原始檔未變動。")
print(f"SHA-256：{output_hash}")
display(saved.head(10))

已核對 demo_orders_clean.csv，共 115 筆；原始檔未變動。
SHA-256：566d03f1701b16128fd119cc2776d775787816fb13f3d7e6d799988f6ce43104


,order_id,order_date,category,quantity,unit_price_twd,order_amount_twd
0,D0001,2026-09-01,A,1,80,80
1,D0002,2026-09-02,B,3,97,291
2,D0003,2026-09-03,C,5,114,570
3,D0004,2026-09-04,A,2,131,262
4,D0005,2026-09-05,B,4,148,592
5,D0006,2026-09-06,C,1,165,165
6,D0007,2026-09-07,A,3,182,546
7,D0008,2026-09-08,B,5,199,995
8,D0010,2026-09-10,A,4,233,932
9,D0011,2026-09-11,B,1,250,250
